# Optimization for Data Science — 2026/2027
## Project: Optimization with imperfect information

**Student version**

Read `ProjODS_2627.pdf` first. The PDF defines the questions and budget convention. This notebook supplies deterministic data, objective-value oracles, the method framework and plots. Complete all `TODO` blocks, keep the prescribed budgets, and write your theoretical answers in the report (English or French).
 **No GPU** is required.

In [ ]:
import sys, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import expit
from scipy.optimize import minimize

print('Python:',sys.version.split()[0], '| NumPy:',np.__version__)

SEED = 202700 

## 1 — Toy noisy oracle (Questions 1–2)

**Before executing the experiments**, predict the shapes of MSE-versus-`h` curves for forward and central differences, with and without noise. Write and justify the predictions in the report. Derive the exact formulas in the PDF, including the quartic counterexample.

*Your predictions (enter here or in the report):* ...

In [ ]:
# A deterministic numerical test of your analytic MSE, using fresh noise at each call.
tau = 0.025
w, a = 0.6, -0.2
hs = np.geomspace(1e-3, 1.2, 24)
theory_forward = hs**2 / 4 + 2*tau**2 / hs**2
theory_central = tau**2 / (2*hs**2)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.loglog(hs, theory_forward, label='forward difference: exact quadratic MSE')
ax.loglog(hs, theory_central, label='central difference: exact quadratic MSE')
ax.axvline((8*tau*tau)**0.25, linestyle=':', label='optimal forward h')
ax.set(xlabel='Finite-difference scale h', ylabel='Mean squared error', title='Analytical curves (toy quadratic)')
ax.legend(); ax.grid(alpha=.2); plt.show()

## 2 — Binary logistic classification (Question 3)

The **same** data generator, split, initial point and regularization parameter will be used for all methods. Labels are sampled from a logistic teacher model. The generator is supplied, so the project focuses on optimization rather than data engineering.

In [ ]:
def make_data(group_seed, n=400, ntest=400, d=8):
    # Nested comparisons: the first 8 features and all labels are IDENTICAL
    # between d=8 and d=24; the remaining 16 features are nuisance variables.
    assert 1 <= d <= 24
    full_d=24
    rng = np.random.default_rng(group_seed + 31*full_d)
    Q, _ = np.linalg.qr(rng.normal(size=(full_d,full_d)))
    scales = np.geomspace(1.0, 0.25, full_d)
    def draw(m):
        Z = rng.standard_normal((m,full_d))
        return (Z * scales) @ Q.T
    Xfull, Xtfull = draw(n), draw(ntest)
    # Fixed scaling from the TRAINING distribution, shared by both d choices.
    scaling = np.sqrt(np.mean(np.sum(Xfull[:,:8]**2,axis=1))/8)
    Xfull, Xtfull = Xfull/scaling, Xtfull/scaling
    w_star8 = rng.normal(size=8)
    w_star8 *= 2.5 / np.linalg.norm(w_star8)
    y = np.where(rng.random(n) < expit(Xfull[:,:8]@w_star8), 1., -1.)
    yt = np.where(rng.random(ntest) < expit(Xtfull[:,:8]@w_star8), 1., -1.)
    return Xfull[:,:d], y, Xtfull[:,:d], yt

X,y,Xt,yt = make_data(SEED,d=8)
lam=0.10
print('Data:', X.shape, Xt.shape, '| label mean:', round(float(y.mean()),3))

In [ ]:
def logistic_objective(w, X, y, lam):
    # TODO: numerically stable logistic loss + L2 regularizer.
    raise NotImplementedError('TODO: logistic_objective')

def logistic_gradient(w, X, y, lam):
    # TODO: exact gradient, including L2 regularizer.
    raise NotImplementedError('TODO: logistic_gradient')

def minibatch_gradient(w, X, y, lam, batch_indices):
    # TODO: use only the selected examples (sampled with replacement).
    raise NotImplementedError('TODO: minibatch_gradient')

In [ ]:
# Diagnostic finite difference checks: this is allowed BEFORE optimization runs.
rng_check = np.random.default_rng(SEED + 111)
wcheck = rng_check.normal(size=X.shape[1]) * 0.3
hh=1e-6
fd=np.array([(logistic_objective(wcheck+hh*np.eye(len(wcheck))[j],X,y,lam)
             -logistic_objective(wcheck-hh*np.eye(len(wcheck))[j],X,y,lam))/(2*hh)
            for j in range(len(wcheck))])
err=np.linalg.norm(fd-logistic_gradient(wcheck,X,y,lam))
print('Finite-difference derivative sanity check:',err)
assert err < 1e-5, 'Check gradient sign and regularization'

## 3 — Two black-box gradient estimates (Question 4)

`ValueOracle.value(w)` represents a **function evaluation only**. Each query has fresh independent noise. Neither CFD nor SPSA may access `logistic_gradient` internally. In a diagnostic experiment only, you may compare their returned estimates to the exact gradient.

In [ ]:
class ValueOracle:
    def __init__(self, X, y, lam, sigma=0., seed=0):
        self.X, self.y, self.lam = X, y, lam
        self.sigma = float(sigma)
        self.rng = np.random.default_rng(seed)
        self.calls = 0
    def value(self, w):
        self.calls += 1
        return logistic_objective(w,self.X,self.y,self.lam) + self.rng.normal(0., self.sigma)

In [ ]:
def coordinate_fd(oracle, w, h):
    # TODO: central finite differences. Exactly 2*d NEW oracle calls.
    raise NotImplementedError('TODO: coordinate_fd')

def spsa(oracle, w, h, rng):
    # TODO: random +/-1 direction; exactly 2 NEW oracle calls.
    raise NotImplementedError('TODO: spsa')

In [ ]:
# Required assertion: both estimates must use ONLY value queries.
o=ValueOracle(X,y,lam,sigma=0.,seed=SEED+101)
gc=coordinate_fd(o,np.zeros(X.shape[1]),h=0.02)
assert o.calls==2*X.shape[1] and gc.shape==(X.shape[1],)
o=ValueOracle(X,y,lam,sigma=0.,seed=SEED+102)
gs=spsa(o,np.zeros(X.shape[1]),h=0.02,rng=np.random.default_rng(SEED+103))
assert o.calls==2 and gs.shape==(X.shape[1],)
print('Black-box call count checks passed.')

## 4 — Oracle accuracy diagnostic (Implementation 2, Question 5)

Estimate squared gradient error at a fixed, nonzero `w`, repeating estimates with independent random streams. For logistic regression, finite-difference errors can contain a nonzero deterministic bias. Discuss *why* a plot may look different from the quadratic one-dimensional example.

In [ ]:
def gradient_error_diagnostic(X,y,lam,sigmas=(0.,0.01),hs=(0.002,0.01,0.04,0.15),repeats=30):
    d=X.shape[1]
    rng=np.random.default_rng(SEED+114)
    w=rng.normal(size=d)*0.3
    truth=logistic_gradient(w,X,y,lam)  # Diagnostic ONLY
    records=[]
    for sigma in sigmas:
        for h in hs:
            for method in ('CFD','SPSA'):
                errs=[]
                for trial in range(repeats):
                    oracle=ValueOracle(X,y,lam,sigma,seed=SEED+700000+trial+1000*int(h*1e6)+100000*int(sigma*1000))
                    est=(coordinate_fd(oracle,w,h) if method=='CFD' else
                         spsa(oracle,w,h,np.random.default_rng(SEED+5678+trial)))
                    errs.append(np.sum((est-truth)**2))
                records.append((method,sigma,h,float(np.mean(errs)),float(np.std(errs))))
    return records
records=gradient_error_diagnostic(X,y,lam)
fig,axes=plt.subplots(1,2,figsize=(11,3.7),sharey=True)
for ax,sigma in zip(axes,(0.,0.01)):
    for method in ('CFD','SPSA'):
        sub=[r for r in records if r[0]==method and r[1]==sigma]
        ax.loglog([r[2] for r in sub],[r[3] for r in sub],marker='o',label=method)
    ax.set(xlabel='Perturbation scale h',title=f'Oracle noise sigma={sigma}')
    ax.grid(alpha=.2);ax.legend()
axes[0].set_ylabel('Mean squared gradient error')
plt.tight_layout();plt.show()
print('method, sigma, h, mean MSE, std:',records[:3])

## 5 — Fair-budget training (Implementation 3, Questions 6–8)

**Before viewing the next curves, record two predictions in the report.** A shared nominal work budget is enforced in **sample-term evaluations**. This is only a transparent accounting proxy and is not the same as elapsed time. The reference optimizer (SciPy) is used *only to compute the reference value for plotting*, not as a candidate method or hyperparameter-selection criterion. Test data must not be used to tune stepsizes or finite-difference scales.

A method may have no more than `budget` nominal sample-term evaluations. Each logged diagnostic clean loss is excluded from the training budget (and explicitly not used for updates). Timed kernel execution also excludes diagnostic plotting work.

In [ ]:
def run_method(method,X,y,lam,budget,seed,alpha,h=0.03,sigma=0.,batch_size=32,
               reference=None, Xt=None, yt=None, record_every=10):
    # TODO: implement the four update rules and common sample-term budget.
    # Expected per step costs: GD=n, SGD=b, CFD=2*d*n, SPSA=2*n.
    # Record clean objective (for reporting only), nominal effort, and
    # elapsed time spent in updates (excluding plotting/clean diagnostics).
    # Do NOT let CFD or SPSA access logistic_gradient in any update.
    raise NotImplementedError('TODO: run_method')

In [ ]:
def reference_optimum(X,y,lam):
    d=X.shape[1]
    res=minimize(logistic_objective,np.zeros(d),args=(X,y,lam),
                 jac=logistic_gradient,method='L-BFGS-B',options={'gtol':1e-11,'maxiter':500})
    assert np.linalg.norm(logistic_gradient(res.x,X,y,lam))<2e-5
    return float(res.fun)

def choose_defaults(X,lam):
    n=X.shape[0]
    L=float(np.linalg.norm(X,ord=2)**2/(4*n)+lam)
    return L,{'GD':1./L,'SGD':0.20/L,'CFD':0.8/L,
              'SPSA':lambda k:0.14/L/np.sqrt(1.+k/40.)}

def trial_suite(d=8,sigma=0.003,budget_passes=240,replicates=3,h=0.04):
    X,y,Xt,yt=make_data(SEED,d=d)
    ref=reference_optimum(X,y,lam)
    L,steps=choose_defaults(X,lam)
    B=budget_passes*len(y)
    outputs=[]
    for method in ('GD','SGD','CFD','SPSA'):
        for rep in range(replicates if (method in ('SGD','SPSA') or (method=='CFD' and sigma>0)) else 1):
            outputs.append(run_method(method,X,y,lam,B,SEED+rep+42*d,
                         alpha=steps[method],h=h,sigma=sigma,batch_size=32,
                         reference=ref,Xt=Xt,yt=yt,record_every=20 if method=='SGD' else 2))
    return outputs

In [ ]:
# These are starting configurations, not mandatory optimal hyperparameters.
# Analyze/justify your choices and improve them using TRAINING data only.
results=trial_suite(d=8,sigma=0.003,budget_passes=240,replicates=3,h=0.04)
fig,axes=plt.subplots(1,2,figsize=(11,3.7))
for r in results:
    label=r['method']
    axes[0].semilogy(r['budget']/400,r['gap'],alpha=.65,label=label)
    axes[1].semilogy(r['seconds'],r['gap'],alpha=.65,label=label)
# Avoid repeated legend labels
for ax in axes:
    handles,labels=ax.get_legend_handles_labels();unique=dict(zip(labels,handles))
    ax.legend(unique.values(),unique.keys());ax.grid(alpha=.2)
axes[0].set(xlabel='Nominal work (full-data passes)',ylabel='Clean objective gap',title='Equal nominal budget')
axes[1].set(xlabel='Cumulative update time (s)',title='Measured update time')
plt.tight_layout();plt.show()
print('Final results:')
for m in ('GD','SGD','CFD','SPSA'):
    sub=[r for r in results if r['method']==m]
    print(f"{m:4} mean_gap={np.mean([r['gap'][-1] for r in sub]):.5f} "
          f"mean_acc={np.mean([r['test_accuracy'] for r in sub]):.3f} "
          f"budget={sub[0]['spent']} steps={sub[0]['steps']} "
          f"mean_time={np.mean([r['kernel_seconds'] for r in sub]):.3f}s")

### Your counter-experiment (Question 8)

Pick one falsifiable claim, state a competing explanation, change **exactly one factor** and repeat the trial with the same group seed, budget and protocol. Suggested choices: `d=8` versus `d=24`, or `sigma=0` versus `sigma=0.003`. Do not change the dataset dimension and the noise at the same time if you want to attribute a difference to one cause.

*Claim: ...*  
*Competing explanation: ...*  
*One-factor intervention: ...*  
*Result and revised conclusion: ...*

In [ ]:
# Example one-factor counter-experiment: change dimension ONLY.
# This cell is an illustrative starting point; justify your own counter-experiment.
results_d24=trial_suite(d=24,sigma=0.003,budget_passes=240,replicates=3,h=0.04)
summary={m: [float(np.mean([r['gap'][-1] for r in rr if r['method']==m]))
             for rr in (results,results_d24)] for m in ('GD','SGD','CFD','SPSA')}
for method,v in summary.items():
    print(f'{method}: objective gap at d=8: {v[0]:.5g}; at d=24: {v[1]:.5g}')

## Submission checklist

- Complete answers to Q1–Q8, including *pre-experiment* predictions.
- Tables with method, learning rate, scale `h`, batch size, budget, dimension and oracle noise.
- Report at least three independent runs where stochasticity is present; describe variability.
- Distinguish nominal work from measured update time.
- One controlled counter-experiment and one paragraph per student about a concrete decision, failure mode or revised interpretation.
- Include seed/group key, versions, a reproducible notebook, and an AI-usage statement.